In [12]:
import TwoPUtils
%load_ext autoreload
%autoreload 2
from reward_relative.path_dict_seahorse import path_dictionary as path_dict
from reward_relative import utilities as ut
from reward_relative import plotUtils as pt
from reward_relative import spatial
from reward_relative import placeCellPlot
from reward_relative import dayData as dd
from reward_relative import behavior
import pickle
import dill
import numpy as np
import os 
import matplotlib.pyplot as plt
import TwoPUtils
import protter_functions as pf
from scipy.signal import medfilt
from matplotlib.lines import Line2D
import pandas as pd
import umap
import hdbscan
import importlib
import h5py
import traceback

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:

experiment = 'MetaLearn'
year = 'combined'
exp_days = [3, 5, 7, 8, 10, 12, 14]

max_anim_list = dd.max_anim_list(experiment, exp_days, year=year)

## These parameters were used for computing the saved multiDayData
# bin_size = 10  # for quantifying distribution of place field peak locations
# sigma = 1  # for smoothing
# smooth = False  # whether to smooth for finding place cell peaks
# exclude_int = True  # exclude putative interneurons
# int_thresh = 0.5

## Place cell logical definitions:
## 'and' = must have significant spatial information
## in trial set 0 AND trial set 1 (i.e. before and after the reward switch)
## 'or' = must have signitive spatial information in trial set 0 OR trial set 1

# place_cell_logical = 'or'
ts_key = 'dff'  # which timeseries to use for finding peaks
# use_speed_thr = True  # use a speed threshold to calculate new trial matrices
# # speed threshold in cm/s (excludes data at speed less than this)
# speed_thr = 2

reward_dist_inclusive = 50  # in cm

# datetime of saved file
dt = "202504"

pkl_name = "%s_expdays%s_multiDayData_%s_%s.pickle" % (
    # ut.make_anim_tag(max_anim_list),
    f'm{ut.get_mouse_number(max_anim_list[0])}-{ut.get_mouse_number(max_anim_list[-1])}',
    ut.make_day_tag(
        exp_days),
    ts_key,
    dt)
pkl_path = os.path.join(
    path_dict['preprocessed_root'], 'multi_anim_sess', pkl_name)
print(pkl_path)
multiDayData = dill.load(open(pkl_path, "rb"), ignore=True)

include_ans = multiDayData[exp_days[-1]
                           ].circ_rel_stats_across_an['include_ans']
max_anim_list = sorted(np.unique(np.concatenate([multiDayData[day].anim_list
                                                 for day in exp_days])),
                       key=len)
include_ans

/data/2p_data/multi_anim_sess/m2-19_expdays3-5-7-8-10-12-14_multiDayData_dff_202504.pickle


array(['GCAMP3', 'GCAMP4', 'GCAMP7', 'GCAMP11', 'GCAMP12', 'GCAMP13',
       'GCAMP14', 'GCAMP15', 'GCAMP17', 'GCAMP18', 'GCAMP19'], dtype='<U7')

In [14]:
include_ans

array(['GCAMP3', 'GCAMP4', 'GCAMP7', 'GCAMP11', 'GCAMP12', 'GCAMP13',
       'GCAMP14', 'GCAMP15', 'GCAMP17', 'GCAMP18', 'GCAMP19'], dtype='<U7')

In [15]:
file_name = '/data/dave/sosa_data_h5/hdf5_conversion'

animal_list = include_ans

if os.path.exists(file_name):
    response = input(f'file already exists! want to overwrite? (y)')
    if not response.lower() == 'y':
        raise FileExistsError('not going to overwrite your db, sorry!')




with h5py.File(file_name, "w") as f:

    animal_groups = {animal: f.create_group(animal) for animal in include_ans}


    for i, day in enumerate(sorted(exp_days)):
        print(day)
        # load multi_anim_sess that has the spatial activity to plot
        multi_anim_sess = dd.load_multi_anim_sess(path_dict, day, multiDayData[day].anim_list,
                                            subdirs = ['multi_anim_sess'])
        
        #some lazy error handling here so I can go home and run this 
        try:
            for animal in animal_list:
                today = animal_groups[animal].create_group(str(day))
                meta = today.create_group('trial_metadata')

                session = multi_anim_sess[animal]['sess']
                
                trial_info = pf.TrialInfo({day:session}).lookup_df
                licks = pf.correct_licks(session)
                licks_smoothed = pf.smooth_trial_matrix(licks)

                data, occupancy, bin_edges, bin_centers = licks

                lick_g = today.create_group('lick_data')

                lick_g.create_dataset('corrected_licks', data = data)
                lick_g.create_dataset('bin_center', data = bin_centers)
                lick_g.create_dataset('smoothed_licks', data = licks_smoothed)
                lick_g.create_dataset('occupancy', data = occupancy)
                lick_g.create_dataset('bin_edges', data = bin_edges)
                


                for key in ['speed', 'rewards']:
                    try:
                        '''unpack trial matrices and put them in their own group. i think this will make code more readable.'''
                        today_k = today.create_group(key)
                        data, occupancy, bin_edges, bin_centers = session.trial_matrices[key]

                        today_k.create_dataset(key, data = data)
                        today_k.create_dataset('bin_center', data = bin_centers)
                        today_k.create_dataset('occupancy', data = occupancy)
                        today_k.create_dataset('bin_edges', data = bin_edges)
                    except:
                        print(f'{key} : {animal} : {day} ')
                
                for col in trial_info.columns:
                    data = trial_info[col].values

                    meta.create_dataset(col, data = data)

                bin_size = 1
                speed_thresh = 2
                impute_nans = True
                
                tm = TwoPUtils.spatial_analyses.trial_matrix(session.timeseries['events'].T,
                                                    session.vr_data['pos']._values,
                                                    session.trial_start_inds,
                                                    session.teleport_inds,
                                                    bin_size=bin_size,
                                                    min_pos=0,
                                                    max_pos=450,
                                                    speed_thr=speed_thresh,
                                                    speed=session.vr_data['speed']._values,
                                                    impute_nans = impute_nans,
                                                    
                                                    )
                
                ds = today.create_dataset('neural_events_trial_matrix', data = tm[0])
                ds.attrs['bin_size'] = bin_size
                ds.attrs['speed_thres'] = speed_thresh
                ds.attrs['impute_nans']= impute_nans
                
        except:
            traceback.print_exc()
            
        del(multi_anim_sess)
        del(session)

3


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

5


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

7


...

/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

12


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

14


/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos/TwoPUtils/TwoPUtils/spatial_analyses.py:73: RuntimeWarning: Mean of empty slice
  trial_mat[trial, b] = np.nanmean(arr_t[(pos_t > edge1) & (pos_t <= edge2), :], axis=0)
/home/sosalab/local_repos

In [16]:
f= h5py.File(file_name, "r")

In [17]:
f.close()

In [18]:
day = 3
multi_anim_sess = dd.load_multi_anim_sess(path_dict, day, multiDayData[day].anim_list,
                                            subdirs = ['multi_anim_sess'])

In [19]:
len(multi_anim_sess['GCAMP7']['sess'].trial_matrices['rewards'])

4

In [20]:
multi_anim_sess['GCAMP14']['sess'].trial_matrices.keys()

dict_keys(['licks', 'rewards', 'speed', 'events', 'dff', 'spks', 'spks_norm'])